In [ ]:
!pip install cohere --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.0/352.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 72.8 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
from datasets import load_dataset
import pandas as pd
import cohere

In [ ]:
login("your-key")
co = cohere.ClientV2("your-api")

In [ ]:
dataset = load_dataset("cais/mmlu",'elementary_mathematics')

README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

elementary_mathematics/test-00000-of-000(…):   0%|          | 0.00/41.1k [00:00<?, ?B/s]

elementary_mathematics/validation-00000-(…):   0%|          | 0.00/9.38k [00:00<?, ?B/s]

elementary_mathematics/dev-00000-of-0000(…):   0%|          | 0.00/4.55k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/378 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/41 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
prompt_COT_fa = f'''
به پرسش زیر قدم به قدم فکر کنید و زنجیره افکار (chain of thought) خود برای رسیدن به پاسخ را به طور کامل شرح دهید و تنها یک گزینه را انتخاب کنید.

## پرسش
{{question}}

## گزینه ها
[A] : {{option1}}
[B] : {{option2}}
[C] : {{option3}}
[D] : {{option4}}
'''

prompt_COT_en = f'''
Think about the following question step-by-step, explain your chain of thought for reaching the answer in detail, and select only one option.

## Question
{{question}}

## Options
[A] : {{option1}}
[B] : {{option2}}
[C] : {{option3}}
[D] : {{option4}}
'''

In [ ]:
def append_record_to_excel(file_path, Question,
                           correct_answer, model_prompt, AI_answer):
    new_record = {
        'Question': Question,
        'correct_answer': correct_answer,
        'model_prompt':  model_prompt,
        'AI_answer': AI_answer
    }
    new_record_df = pd.DataFrame([new_record])
    try:
        existing_df = pd.read_excel(file_path)
        updated_df = pd.concat([existing_df, new_record_df], ignore_index=True)
    except FileNotFoundError:
        updated_df = new_record_df

    updated_df.to_excel(file_path, index=False)

def convert_to_letters(number):
    if number == 0:
        return 'A'
    elif number == 1:
        return 'B'
    elif number == 2:
        return 'C'
    elif number == 3:
        return 'D'
    else:
        return None

def extract_answer(obj):
    return str(obj.message.content[0]).split("text=")[1][1:-1].replace("\\n","\n")

In [ ]:
EXP_NUM = 9
counter = 0
for example in dataset["test"]:
    prmpt = prompt_COT_en.format(question=example["question"], option1=example["choices"][0], option2=example["choices"][1], option3=example["choices"][2], option4=example["choices"][3])
    obj = co.chat(
    messages=[{"role": "user", "content": prmpt}],
    temperature=1.0,
    model="c4ai-aya-expanse-32b")

    file_path = 'aya-expanse-32b_elementary_mathematics_En_no'+str(EXP_NUM)+'.xlsx'
    append_record_to_excel(file_path = file_path,Question=example["question"],
                           correct_answer=convert_to_letters(example["answer"]),
                           model_prompt = prmpt, AI_answer=extract_answer(obj))
    counter += 1
    print("Question number " + str(counter) + " has been solved.")
    print('================================================================')

Question number 1 has been solved.
Question number 2 has been solved.
Question number 3 has been solved.
Question number 4 has been solved.
Question number 5 has been solved.
Question number 6 has been solved.
Question number 7 has been solved.
Question number 8 has been solved.
Question number 9 has been solved.
Question number 10 has been solved.
Question number 11 has been solved.
Question number 12 has been solved.
Question number 13 has been solved.
Question number 14 has been solved.
Question number 15 has been solved.
Question number 16 has been solved.
Question number 17 has been solved.
Question number 18 has been solved.
Question number 19 has been solved.
Question number 20 has been solved.
Question number 21 has been solved.
Question number 22 has been solved.
Question number 23 has been solved.
Question number 24 has been solved.
Question number 25 has been solved.
Question number 26 has been solved.
Question number 27 has been solved.
Question number 28 has been solved.
Q